<div style="font-size:2em; font-weight:bold; margin-bottom:8px;">07 — Index Health Report</div>

This notebook is the final step of the pipeline. It checks that the Qdrant index built in notebook 06 is healthy and demonstrates a real end-to-end query: text question in, relevant chunks out.

It is **Step 7** of the RAG data indexing pipeline — verification and reporting only.

---

**What this notebook does:**
1. Connects to Qdrant
2. Reads the collection status (point count, vector size, distance)
3. Loads the embedding model so it can embed a text question
4. Runs an end-to-end search: embed a question, retrieve the top matching chunks
5. Runs a metadata-filtered search
6. Prints a final health report matching the project goal's status shape

**What this notebook intentionally does NOT do:**
- No re-indexing (that is notebook 06)
- No answer generation — this repository stops at retrieval-ready chunks

> **Before running:** notebook 06 must have indexed the chunks into Qdrant, and
> Qdrant must be running (`make up`).

---
## 1. Imports

We need:
- **`os`** — read Qdrant connection settings from the environment
- **`SentenceTransformer`** — embed the text question with the same model used for indexing
- **`QdrantClient`** and **`models`** — connect to Qdrant and build the metadata filter

In [2]:
# ── [1 / 8] Imports ─────────────────────────────────────────────────────────

import os

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

print("Imports ready.")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports ready.


---
## 2. Configuration

The same settings used in notebook 06, read from the environment when available.
`EMBEDDING_MODEL` must match the model used in notebook 05, otherwise the query
vector would live in a different space than the indexed vectors.

In [3]:
# ── [2 / 8] Configuration ───────────────────────────────────────────────────

QDRANT_URL      = os.environ.get("QDRANT_URL", "http://qdrant:6333")
COLLECTION      = os.environ.get("QDRANT_COLLECTION", "rag_scifact")
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5")

# A natural-language question to test retrieval against the SciFact corpus
QUESTION = "How does diffusion tensor MRI reveal white matter development in infants?"
TOP_K    = 5

print(f"Qdrant URL      : {QDRANT_URL}")
print(f"Collection      : {COLLECTION}")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Question        : {QUESTION}")

Qdrant URL      : http://qdrant:6333
Collection      : rag_scifact
Embedding model : BAAI/bge-small-en-v1.5
Question        : How does diffusion tensor MRI reveal white matter development in infants?


---
## 3. Connect and Read Collection Status

We connect to Qdrant and read the collection info: how many points it holds, the
vector size, and the distance metric. If the collection is missing, we stop with a
clear message to run notebook 06 first.

In [4]:
# ── [3 / 8] Connect and Read Collection Status ──────────────────────────────

client = QdrantClient(url=QDRANT_URL, timeout=30)

if not client.collection_exists(COLLECTION):
    raise RuntimeError(
        f"Collection '{COLLECTION}' does not exist.\n"
        "Run notebook 06_index_qdrant.ipynb first."
    )

info  = client.get_collection(COLLECTION)
count = client.count(collection_name=COLLECTION, exact=True).count
vector_params = info.config.params.vectors

print(f"Collection    : {COLLECTION}")
print(f"Points stored : {count:,}")
print(f"Vector size   : {vector_params.size}")
print(f"Distance      : {vector_params.distance.name}")
print(f"Status        : {info.status}")

Collection    : rag_scifact
Points stored : 12,281
Vector size   : 384
Distance      : COSINE
Status        : green


---
## 4. Load the Embedding Model

To search by a text question, we must embed the question with the **same** model
used to build the index. We also normalize the query vector, matching how the
chunks were embedded in notebook 05.

In [5]:
# ── [4 / 8] Load the Embedding Model ────────────────────────────────────────

print(f"Loading model: {EMBEDDING_MODEL} ...")
model = SentenceTransformer(EMBEDDING_MODEL)

# Confirm the model dimension matches what the collection expects
assert model.get_sentence_embedding_dimension() == vector_params.size, \
    "embedding model dimension does not match the collection vector size"
print("Model loaded and dimension matches the collection.")

Loading model: BAAI/bge-small-en-v1.5 ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1455.75it/s]


Model loaded and dimension matches the collection.


/tmp/ipykernel_425/3077204802.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert model.get_sentence_embedding_dimension() == vector_params.size, \


---
## 5. End-to-End Query Demo

This is the payoff: we embed the natural-language `QUESTION`, search Qdrant, and
print the top matching chunks with their score and metadata. This is exactly what
a downstream retrieval service would do to gather context for an answer.

In [6]:
# ── [5 / 8] End-to-End Query Demo ───────────────────────────────────────────

query_vector = model.encode(QUESTION, normalize_embeddings=True).tolist()

hits = client.query_points(
    collection_name=COLLECTION,
    query=query_vector,
    limit=TOP_K,
    with_payload=True,
).points

print(f"Question: {QUESTION}")
print("=" * 70)
for rank, hit in enumerate(hits, start=1):
    title = (hit.payload.get("title") or "")[:65]
    text  = (hit.payload.get("text")  or "").replace("\n", " ")[:140]
    print(f"\n[{rank}] score={hit.score:.4f}  doc={hit.payload.get('document_id')}")
    print(f"    title: {title}")
    print(f"    text : {text}...")

assert len(hits) > 0, "search returned no results — is the collection empty?"

Question: How does diffusion tensor MRI reveal white matter development in infants?

[1] score=0.9022  doc=4983
    title: Microstructural development of human newborn cerebral white matte
    text : 33.1 +/- 0.6% p = 0.006). Nonmyelinated fibers in the corpus callosum were visible by diffusion tensor MRI as early as 28 wk; full-term and ...

[2] score=0.8813  doc=4983
    title: Microstructural development of human newborn cerebral white matte
    text : Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functio...

[3] score=0.8414  doc=4983
    title: Microstructural development of human newborn cerebral white matte
    text : at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean appa...

[4] score=0.8089  doc=19685306
    title: Orientationally invariant indices of axon diameter and density fr
    text : This paper propo

---
## 6. Metadata-Filtered Query

We repeat the query but restrict it to chunks whose `document_type` matches what we
set in notebook 04. This confirms metadata filtering works end-to-end.

In [7]:
# ── [6 / 8] Metadata-Filtered Query ─────────────────────────────────────────

DOCUMENT_TYPE = "scientific_abstract"

filtered = client.query_points(
    collection_name=COLLECTION,
    query=query_vector,
    query_filter=models.Filter(
        must=[models.FieldCondition(
            key="document_type",
            match=models.MatchValue(value=DOCUMENT_TYPE),
        )]
    ),
    limit=TOP_K,
    with_payload=True,
).points

print(f"Filtered query (document_type == '{DOCUMENT_TYPE}') returned {len(filtered)} hits.")
for rank, hit in enumerate(filtered, start=1):
    print(f"  {rank}. score={hit.score:.4f}  doc={hit.payload.get('document_id')}")

assert all(h.payload.get("document_type") == DOCUMENT_TYPE for h in filtered), \
    "filter returned a chunk with the wrong document_type"
print("\nMetadata filter verified.")

Filtered query (document_type == 'scientific_abstract') returned 5 hits.
  1. score=0.9022  doc=4983
  2. score=0.8813  doc=4983
  3. score=0.8414  doc=4983
  4. score=0.8089  doc=19685306
  5. score=0.8070  doc=25789730

Metadata filter verified.


---
## 7. Count Indexed Documents

The collection stores chunks, not whole documents. To report how many distinct
documents are indexed, we scroll through the points and collect the unique
`document_id` values.

In [8]:
# ── [7 / 8] Count Indexed Documents ─────────────────────────────────────────

unique_docs = set()
next_offset = None

while True:
    records, next_offset = client.scroll(
        collection_name=COLLECTION,
        limit=1000,
        offset=next_offset,
        with_payload=["document_id"],   # only fetch the field we need
        with_vectors=False,
    )
    for rec in records:
        unique_docs.add(rec.payload.get("document_id"))
    if next_offset is None:
        break

print(f"Distinct documents indexed : {len(unique_docs):,}")
print(f"Total chunks indexed       : {count:,}")

Distinct documents indexed : 5,183
Total chunks indexed       : 12,281


---
## 8. Final Health Report

We assemble a single health report dictionary in the same shape as the collection
status response described in the project goal. If you can read this report, the
whole pipeline — load, clean, chunk, enrich, embed, index — worked end to end.

In [9]:
# ── [8 / 8] Final Health Report ─────────────────────────────────────────────

import json  # local import keeps this summary cell self-contained

health_report = {
    "collection_name"   : COLLECTION,
    "vectors_count"     : count,
    "indexed_documents" : len(unique_docs),
    "vector_size"       : vector_params.size,
    "distance"          : vector_params.distance.name,
    "embedding_model"   : EMBEDDING_MODEL,
    "status"            : str(info.status),
    "search_ok"         : len(hits) > 0,
    "filter_ok"         : len(filtered) > 0,
}

print("=" * 60)
print("INDEX HEALTH REPORT")
print("=" * 60)
print(json.dumps(health_report, indent=2))
print()

if health_report["vectors_count"] > 0 and health_report["search_ok"]:
    print("HEALTHY — the index is populated and searchable.")
else:
    print("UNHEALTHY — re-run notebooks 05 and 06.")

INDEX HEALTH REPORT
{
  "collection_name": "rag_scifact",
  "vectors_count": 12281,
  "indexed_documents": 5183,
  "vector_size": 384,
  "distance": "COSINE",
  "embedding_model": "BAAI/bge-small-en-v1.5",
  "status": "green",
  "search_ok": true,
  "filter_ok": true
}

HEALTHY — the index is populated and searchable.
